## 캐싱(Caching)

LLM Cache는 LLM 호출 결과를 저장합니다.

LangChain은 LLM을 위한 선택적 캐싱 레이어를 제공합니다.

이는 두 가지 이유로 유용합니다.

- 동일한 완료를 여러 번 요청하는 경우 LLM 공급자에 대한 **API 호출 횟수를 줄여 비용을 절감**할 수 있습니다.
- LLM 제공업체에 대한 **API 호출 횟수를 줄여 애플리케이션의 속도를 높일 수** 있습니다.

In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH04-Models")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH04-Models


모델과 프롬프트를 생성합니다


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# 모델을 생성합니다.
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template("{country} 에 대해서 200자 내외로 요약해줘")

# 체인을 생성합니다.
chain = prompt | llm

In [ ]:
# %%time : 해당 셀의 코드가 실행되는데 걸린 시간을 측정
%%time 
response = chain.invoke({"country": "한국"})
print(response.content)

한국은 동아시아의 한반도에 위치한 민주공화국으로, 수도는 서울에 있다. 역사적으로는 다른 나라들의 침략과 식민지배를 겪었지만 지금은 현대화된 산업과 문화로 발전하고 있다. 한국은 IT 산업, 자동차 제조업, K-pop과 같은 문화 콘텐츠 등에서 세계적인 영향력을 가지고 있으며, 한식과 한복 등 전통문화도 인기가 높다. 한국은 현대와 전통이 어우러진 독특한 매력을 가지고 있으며, 빠르게 변화하는 사회와 고유한 문화가 외국인들에게 큰 관심을 끌고 있다.이 외에도 한국은 미식 경험과 아름다운 자연경관, 역사적인 유적지와 문화예술 등 다양한 매력을 가지고 있는 나라이다.
CPU times: total: 141 ms
Wall time: 4.32 s


## InMemoryCache

인메모리 캐시를 사용하여 동일 질문에 대한 답변을 저장하고, 캐시에 저장된 답변을 반환합니다.

In [ ]:
%%time
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache

# 인메모리 캐시를 사용합니다.
# InMemoryCache 객체를 생성하여 set_llm_cache에 전달하고 Langchain의 LLM 캐시로 등록하면 
# chain.invoke()에서 캐시를 사용함
set_llm_cache(InMemoryCache())

# 체인을 실행하면 설정해 놓은 InMemoryCache 사용
response = chain.invoke({"country": "한국"})
print(response.content)

한국은 동아시아에 위치한 고도로 발전한 현대화된 나라이다. 경제적으로 선진국에 속하며 기술력과 문화적 발전이 뛰어나다. 한반도에 위치하고 있어 북한과 이웃해 있는데, 남한과 더불어 건강한 경제활동을 펼친다. 한국은 고대부터 다양한 역사와 전통을 갖고 있으며, 한류 문화로 전 세계에 대중적인 영향력을 행사하고 있다. 주요 도시로는 수도인 서울과 광역시인 부산, 대구, 인천 등이 있다. 또한 한반도 자연환경도 아름답고 다양하다. 식품, 자동차, 전자제품 등 다양한 산업이 발달하고 있으며, K-POP, K-드라마 등을 통해 문화 콘텐츠 산업도 유명하다.
CPU times: total: 15.6 ms
Wall time: 4.22 s


In [9]:
%%time
# 체인을 실행합니다.
response = chain.invoke({"country": "한국"})
print(response.content)

한국은 동아시아에 위치한 고도로 발전한 현대화된 나라이다. 경제적으로 선진국에 속하며 기술력과 문화적 발전이 뛰어나다. 한반도에 위치하고 있어 북한과 이웃해 있는데, 남한과 더불어 건강한 경제활동을 펼친다. 한국은 고대부터 다양한 역사와 전통을 갖고 있으며, 한류 문화로 전 세계에 대중적인 영향력을 행사하고 있다. 주요 도시로는 수도인 서울과 광역시인 부산, 대구, 인천 등이 있다. 또한 한반도 자연환경도 아름답고 다양하다. 식품, 자동차, 전자제품 등 다양한 산업이 발달하고 있으며, K-POP, K-드라마 등을 통해 문화 콘텐츠 산업도 유명하다.
CPU times: total: 0 ns
Wall time: 1.31 ms


## SQLite Cache

- SQLite : SQL을 사용하는 경량 관계형 데이터베이스

llm은 기본적으로 텍스트를 입력(prompt)받고 텍스트를 출력(reponse)하는 모델입니다. 즉, 데이터의 구조가 단순하고 변하지 않기 떄문에 SQLite 같은 관계형 DB를 사용합니다.

특히, SQLite은 서버를 따로 설치하거나 실행할 필요 없이 `.db` 파일 하나로 사용할 수 있다는 장점이 있어 로컬 캐시 용도에 잘 맞습니다.

데이터베이스 쿼리 속도를 높이고 반복적인 디스크 I/O를 줄이기 위해 메모리나 디스크에 데이터를 임시 저장하는 기법입니다.


In [ ]:
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache
import os

# 캐시 디렉토리를 생성합니다.
if not os.path.exists("cache"):
    os.makedirs("cache")

# LLM 캐시를 SQLiteCache로 설정
# llm의 요청과 응답을 캐시(저장)하는 코드
set_llm_cache(SQLiteCache(database_path="cache/llm_cache.db"))

C:\Users\HAN\AppData\Local\Temp\ipykernel_5212\1048111182.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.cache import SQLiteCache


In [ ]:
%%time 
# 체인을 실행합니다.
# 실제 캐시 데이터가 저장됨
response = chain.invoke({"country": "한국"})
print(response.content)

한국은 동아시아에 위치한 대한민국과 조선민주주의인민공화국으로 나뉜다. 대한민국은 수도가 서울에 위치하고 민주주의를 기반으로 한 경제적으로 번영한 국가이며, K-pop, K-drama 등의 문화 콘텐츠로 세계적으로 유명하다. 한국은 기술력과 ICT 분야에서 선진국으로 인정받고 있으며 세계 각국과 교류를 통해 국제적인 영향력을 키우고 있다. 조선민주주의인민공화국은 북한으로 불리며 수도는 평양에 위치해 있으며 단일체제를 유지하고 있다. 한반도 분단 상태로 북한과 남한 사이에 이념적 갈등이 지속되고 있지만 최근 남북관계 변화와 핵문제 등 다양한 이슈로 관심을 끌고 있다.
CPU times: total: 15.6 ms
Wall time: 4.28 s
